In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

import os
from datetime import datetime

from src.config import ProjectPath

In [ ]:
sns.set_theme(style="whitegrid")

data_name = "Brain"
valid_methods = [
    "variance",
    "correlation",
    "chi_squared",
    "mutual_information",
    "anova_f_test",
]
n_features = 50

path = ProjectPath(data_name, n_features)

data_dir = path.filter_dir
result = []

In [ ]:
print("󰚩 Training model...")
for m in valid_methods:
    data_path = f"{data_dir}/Brain_{m}_{n_features}features.csv"

    try:
        # 1. read data
        filtered_df = pd.read_csv(data_path)
        y = filtered_df.iloc[:, 0]
        X = filtered_df.iloc[:, 1:]

        # 2. train/test split 80/20
        X_train_filter, X_test_filter, y_train_filter, y_test_filter = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        # 3. Training 

        # LogisticRegression
        log_model = LogisticRegression(max_iter=10000, random_state=42)
        log_model.fit(X_train_filter, y_train_filter)
        y_pred_log = log_model.predict(X_test_filter)
        log_acc = accuracy_score(y_test_filter, y_pred_log)
        
        # DecisionTree
        tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
        tree_model.fit(X_train_filter, y_train_filter)
        y_pred_tree = tree_model.predict(X_test_filter)
        tree_acc = accuracy_score(y_test_filter, y_pred_tree)

        # 4. result
        result.append({'Method': m.upper(), "Model": 'LogisticRegression', "Acc": log_acc})
        result.append({'Method': m.upper(), "Model": 'DecisionTreeClassifier', "Acc": tree_acc})

        print(f"󰄭  {m.upper():<12}: LogReg = {log_acc:.4f} | Tree = {tree_acc:.4f}") 

    except FileNotFoundError: 
        print(f" Cannot found {data_path}")




## Training with not filterd data

In [ ]:
raw_path = path.raw_path
df = pd.read_csv(raw_path)

In [ ]:
print("󰚩 Training model...")
# train_test_split
y = df.iloc[:, 0]
X = df.iloc[:, 1:]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# training

# LogisticRegression
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)
log_acc = accuracy_score(y_test, y_pred_log)

# DecisionTree
tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)
tree_acc = accuracy_score(y_test, y_pred_tree)

result.append({'Method': 'None', "Model": 'LogisticRegression', "Acc": log_acc})
result.append({'Method':  'None',"Model": 'DecisionTreeClassifier', "Acc": tree_acc})

print(f"󰄭  None method: LogReg = {log_acc:.4f} | Tree = {tree_acc:.4f}") 

## Result

In [ ]:

result_df = pd.DataFrame(result)
display(result_df) #type: ignore

In [ ]:
# setup save
report_dir = str(path.filter_result_dir / "report")
plot_dir = str(path.filter_result_dir / "plot")
os.makedirs(report_dir, exist_ok=True)
os.makedirs(plot_dir, exist_ok=True)

report_path = f"{report_dir}/model_evaluation_top{n_features}.txt"
plot_path = f"{plot_dir}/model_comparison_top{n_features}.png"

## Chart

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=result_df, x='Method', y='Acc', hue='Model', palette='pastel')

plt.title(f'Compare top {n_features} features and All features', fontsize= 14, fontweight='bold')
plt.ylim(0.5, 1.05)
plt.ylabel('Acc')
plt.xlabel("Method")
plt.xticks(rotation=45, ha='right', fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"󰋩 Saved plot at {plot_path}")

plt.show()

In [ ]:
with open(report_path, "w", encoding="utf-8") as f:
    f.write("Model Comparing report \n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"n_features: {n_features}\n")
    f.write("-"*60 + "\n")

    f.write(result_df.to_string(index=False))

print(f"󰎞 Report saved at: {report_path}")